# Threshold Optimization — ACSIncome (Illinois, 2018)

Apply group-specific postprocessing to Logistic Regression and Random Forest using true positive rate parity. Thresholds are fitted on the validation split; the held-out test split is reserved for final evaluation.

## Data and models

Use the shared 60/20/20 stratified split and baseline model pipelines. Fit the classifiers on training data only.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from fairlearn.metrics import false_positive_rate, true_positive_rate
from fairlearn.postprocessing import ThresholdOptimizer
from sklearn.metrics import accuracy_score, balanced_accuracy_score

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_acs_income
from src.experiment import prepare_data_splits
from src.models import create_logistic_regression, create_random_forest

In [ ]:
X, y, A = load_acs_income()
X_train, X_val, X_test, y_train, y_val, y_test, A_train, A_val, A_test = (
    prepare_data_splits(X, y, A)
)

In [ ]:
models = {
    "Logistic Regression": create_logistic_regression(),
    "Random Forest": create_random_forest(),
}

for model in models.values():
    model.fit(X_train, y_train)

## Threshold optimization

Fit the postprocessor on validation data with `true_positive_rate_parity` and `balanced_accuracy_score`. Predictions use a fixed random seed because the fitted postprocessor can randomize decisions.

In [ ]:
optimized_models = {}
for name, model in models.items():
    optimizer = ThresholdOptimizer(
        estimator=model,
        constraints="true_positive_rate_parity",
        objective="balanced_accuracy_score",
        prefit=True,
        predict_method="predict_proba",
    )
    optimizer.fit(X_val, y_val, sensitive_features=A_val)
    optimized_models[name] = optimizer

## Validation metrics

Report Accuracy, Balanced Accuracy, absolute Equal Opportunity Difference (EOD), absolute Demographic Parity Difference (DPD), and absolute FPR Gap. These are in-sample postprocessing results because the thresholds were fitted on the same validation split.

In [ ]:
def evaluate_threshold(name, optimizer):
    predictions = optimizer.predict(
        X_val, sensitive_features=A_val, random_state=42
    )
    male = np.asarray(A_val) == 1
    female = np.asarray(A_val) == 2

    tpr_gap = (
        true_positive_rate(y_val[male], predictions[male])
        - true_positive_rate(y_val[female], predictions[female])
    )
    fpr_gap = (
        false_positive_rate(y_val[male], predictions[male])
        - false_positive_rate(y_val[female], predictions[female])
    )
    positive_rate_gap = np.mean(predictions[male]) - np.mean(predictions[female])

    return {
        "Model": name,
        "Strategy": "Threshold Optimization",
        "Accuracy": accuracy_score(y_val, predictions),
        "Balanced Accuracy": balanced_accuracy_score(y_val, predictions),
        "EOD": abs(tpr_gap),
        "DPD": abs(positive_rate_gap),
        "FPR Gap": abs(fpr_gap),
    }


threshold_results = pd.DataFrame([
    evaluate_threshold(name, optimizer)
    for name, optimizer in optimized_models.items()
])
display(threshold_results.round(4))

,Model,Strategy,Accuracy,Balanced Accuracy,EOD,DPD,FPR Gap
0,Logistic Regression,Threshold Optimization,0.7971,0.7962,0.0015,0.1283,0.0417
1,Random Forest,Threshold Optimization,0.7767,0.7822,0.0014,0.1118,0.0223


## Comparison with previous strategies

Combine the validation results with Baseline and Reweighing. Lower gaps indicate smaller measured group disparities; comparisons involving Threshold Optimization should account for its validation-set fitting.

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)

baseline = pd.read_csv(results_dir / "baseline_comparison.csv")
reweighing = pd.read_csv(results_dir / "reweighing_comparison.csv")
validation_results = pd.concat(
    [baseline, reweighing, threshold_results], ignore_index=True
)
display(validation_results.round(4))

threshold_results.to_csv(results_dir / "threshold_comparison.csv", index=False)
validation_results.to_csv(results_dir / "validation_comparison.csv", index=False)

,Model,Strategy,Accuracy,Balanced Accuracy,EOD,DPD,FPR Gap
0,Logistic Regression,Baseline,0.8020,0.7901,0.0215,0.1405,0.0515
1,Random Forest,Baseline,0.7868,0.7706,0.0294,0.1305,0.0413
2,Logistic Regression,Reweighing,0.7994,0.7870,0.0349,0.0923,0.0075
3,Random Forest,Reweighing,0.7839,0.7666,0.0081,0.1027,0.0203
4,Logistic Regression,Threshold Optimization,0.7971,0.7962,0.0015,0.1283,0.0417
5,Random Forest,Threshold Optimization,0.7767,0.7822,0.0014,0.1118,0.0223


## Interpretation and limitations

Group-specific thresholds use `SEX` at prediction time, unlike the baseline and reweighing models. This raises deployment and policy considerations. Validation results support model selection and diagnostics, not unbiased performance claims; final evaluation uses the untouched test split.